# 01 — Preparação dos dados

Este notebook mostra, passo a passo, como o dataset de fine-tuning é construído:

1. **PubMedQA** (literatura biomédica pública) — 1000 exemplos rotulados por especialistas.
2. **Dados sintéticos do hospital** — protocolos, FAQs, modelos de laudo/receita, recusas de prescrição e casos de paciente.
3. **Anonimização** (PII/PHI) e **curadoria** (duplicados + guardrails) antes de qualquer treino.

Roda em CPU, localmente ou no Colab. Para rodar no Colab, execute antes a célula de setup do notebook 02.

In [ ]:
import json, sys
from pathlib import Path

# permite importar `src` quando o notebook roda dentro da pasta notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("Projeto em:", ROOT)

## 1. PubMedQA — o que tem dentro?

In [ ]:
from collections import Counter
from src.data.pubmedqa import load_pubmedqa

records = load_pubmedqa(ROOT / "data/raw/ori_pqal.json")
print(len(records), "exemplos")
print(Counter(r["decision"] for r in records))
r = records[0]
print("\nPERGUNTA:", r["question"])
print("CONTEXTO (1º trecho):", r["contexts"][0][:300], "...")
print("CONCLUSÃO DOS AUTORES:", r["long_answer"][:300])
print("DECISÃO:", r["decision"])

Repare na distribuição: **~55% yes, ~34% no, ~11% maybe**. Um modelo que respondesse sempre "yes" já acertaria ~55%.
Por isso, na avaliação comparamos o fine-tuned com o modelo base *e* com esse baseline.

## 2. Base de conhecimento sintética do hospital

In [ ]:
from src.knowledge.protocols import PROTOCOLS, HOLDOUT_PROTOCOLS, build_chunks

print(len(PROTOCOLS), "protocolos |", len(build_chunks()), "trechos citáveis")
print("Protocolos reservados só para TESTE (generalização):", sorted(HOLDOUT_PROTOCOLS))
p = PROTOCOLS[0]
print("\n", p["id"], "-", p["title"])
for section, text in p["sections"].items():
    print(f"  [{p['id']}#{section}] {text[:110]}...")

Cada trecho tem um id no formato `[PROTOCOLO#seção]`. É isso que o modelo aprende a **citar**, e o que permite a *explainability*:
qualquer afirmação da resposta pode ser rastreada até o texto original.

> Os protocolos são **fictícios** e propositalmente não contêm doses nem nomes de medicamentos.

## 3. Anonimização

In [ ]:
from src.data.anonymization import anonymize_text

exemplo = ("Paciente João da Silva, CPF 123.456.789-09, prontuário nº 889123, "
           "tel (21) 99999-1234, e-mail joao@email.com, internado em 12/03/2025. PA 180/120 mmHg.")
res = anonymize_text(exemplo)
print("ANTES :", exemplo)
print("DEPOIS:", res.text)
print("Substituições:", dict(res.replacements))

Os valores clínicos (PA, creatinina, doses de referência de protocolos) **não** são alterados — só identificadores pessoais.

## 4. Construindo o dataset (anonimização → curadoria → split)

In [ ]:
from src.data.build_dataset import build

stats = build(out_dir=ROOT / "data/processed", seed=42, max_pubmed_train=500)
print(json.dumps(stats, ensure_ascii=False, indent=2))

**Como ler os números**

- `train_by_kind`: quantos exemplos de cada tipo (PubMedQA vs. hospital: protocol_qa, faq, refusal, template, out_of_scope, patient).
- `curation`: o que a curadoria descartou (duplicados, respostas vazias, respostas-alvo que violariam os guardrails).
- `anonymization_replacements`: vazio aqui porque os dados sintéticos não têm PII — **o passo existe e é testado**, e seria ativado com dados reais.

## 5. Inspecionando exemplos

In [ ]:
from src.utils import read_jsonl

train = read_jsonl(ROOT / "data/processed/train.jsonl")
for kind in ("pubmed_qa", "protocol_qa", "refusal", "patient"):
    ex = next(e for e in train if e["meta"]["kind"] == kind)
    print("=" * 80)
    print("TIPO:", ex["meta"]["source"], "/", kind)
    print("--- USER (início) ---")
    print(ex["messages"][1]["content"][:600], "...")
    print("--- ASSISTANT (alvo do treino) ---")
    print(ex["messages"][2]["content"][:500])

**Ponto central do fine-tuning:** o modelo é treinado para produzir a parte `ASSISTANT` (loss só nela).
O que ele aprende é um *comportamento*: usar apenas os trechos dados, citar `[ID#seção]`, recusar prescrição, admitir quando não há base e lembrar a validação humana.